# Cameo Requirements Extraction + Artifact Search — Chained Jobs

A recipe for uploading a Cameo `.mdzip` file, running an extraction job, and then **searching for a specific artifact by name or keyword** without knowing the model name or artifact IDs up front.

It uses [`istari_fluent`](../fluent), an opinionated, chainable wrapper over the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup).

### What we cover

- Connecting to the platform with a Personal Access Token.
- Registering a Cameo `.mdzip` file as a Model.
- Running an extraction job to produce artifacts.
- **Searching for a specific artifact using two approaches:**
  - Structural filter via `platform.resources()` — for when you know the filename or display name.
  - Full-text search via `platform.client.search_resources()` — for keyword search across all metadata fields.
- Reading the found artifact's content.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the **Cameo** integration and access to `@istari:extract`.
- A Cameo `.mdzip` file to extract from.

### 1 &middot; Credentials

Create a `.env` file **next to this notebook** with:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

### 2 &middot; Install dependencies

```bash
cd fluent
uv sync --extra experiment
```

### 3 &middot; Register the venv as a Jupyter kernel

```bash
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

> **A note on `istari_fluent`** — this is a productivity layer maintained alongside the official SDK. It is not the officially supported client. For production integrations, keep the core [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) as your source of truth.

## 1 &middot; Connect and verify

`IstariPlatform.from_env()` reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` and returns an `IstariPlatform` object.

In [1]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, ResourceView

platform = IstariPlatform.from_env()

# Corporate network with an internal CA bundle? Point from_env() at your .pem:
# platform = IstariPlatform.from_env(ca_bundle="/path/to/ca.pem")

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

/Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/fluent/.venv/lib/python3.11/site-packages/istari_digital_client/log_utils.py:32: UserWarning: SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
  result = func(*args, **kwargs)
2026-05-11 18:25:30 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 &middot; Configure job parameters

Set the path to your Cameo `.mdzip` file, the target Cameo version and OS, and the artifact filename you want to search for after extraction.

To see available functions and supported tool versions on your platform:
```python
functions = platform.client.list_functions(tool="dassault_cameo")
for f in functions.items:
    print(f.name, f.tool_versions, f.operating_systems)
```

In [2]:
# --- Configure these for your environment ---
MDZIP_PATH        = Path.cwd() / "NCXTable-example.mdzip"  # path to your Cameo file
TOOL_VERSION      = "2024x-refresh2"                        # adjust to your Cameo version
OPERATING_SYSTEM  = "Windows 11"                            # adjust to your agent's OS
DISPLAY_NAME      = "NCXTable-example-search.mdzip"
EXTERNAL_ID       = "cameo-extract-and-search-demo"

# Artifact to search for — used in Section 6
SEARCH_FILENAME   = "requirements.json"   # exact filename produced by @istari:extract
SEARCH_KEYWORD    = "requirements"        # keyword for full-text search
# --------------------------------------------

assert MDZIP_PATH.exists(), f"Cameo file not found: {MDZIP_PATH}"
print(f"Using model file: {MDZIP_PATH}")
print(f"Tool version:     {TOOL_VERSION}")
print(f"Operating system: {OPERATING_SYSTEM}")

Using model file: /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/NCXTable-example.mdzip
Tool version:     2024x-refresh2
Operating system: Windows 11


## 3 &middot; Register the Cameo file as a Model Resource

Registering a local file creates a **Resource** of type **model** with a stable identity and revision history.

In [3]:
model = platform.upload_model(
    MDZIP_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded new model {model.id}")
print(model)

Uploaded new model 86a66ac3-3a66-4541-85c5-2fdd94011ec8
Model('NCXTable-example-search.mdzip', filename='NCXTable-example.mdzip', id=86a66ac3-3a66-4541-85c5-2fdd94011ec8, file=2f5cf910-8793-4b7b-830b-660f297f6f62, rev=97265ff8-5766-4c06-b196-0b5bd83af48f)


## 4 &middot; Run the extraction job

A **Job** instructs the platform to run a function from a tool against a Model revision. Here we run `@istari:extract` with the `dassault_cameo` tool.

1. `model.submit_job(definition)` returns a `JobView` immediately — the job is queued.
2. `job.wait(on_poll=...)` blocks until the job reaches a terminal state.
3. `.on_success()` raises `RuntimeError` if the job ended in `FAILED`.

> **Shortcut:** `model.run_job(definition)` submits + waits + checks success in one call.

In [4]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
)

job = model.submit_job(extract)
print(f"Submitted job {job.id}; polling...")

job.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob finished: {job.status}")

Submitted job 60e4ced1-9683-4277-83a1-f2112352ff24; polling...
  [Pending] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Running] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Uploading] id=60e4ced1-9683-4277-83a1-f2112352ff24
  [Completed] id=60e4ced1-9683-4277-83a1-f2112352ff24

Job finished: Completed


## 5 &middot; Inspect the job products

`job.get_products()` returns `ResourceView` objects pinned to the exact revisions the job wrote. This is the direct path when you already have a `JobView` in hand.

In [ ]:
products = job.get_products()
print(f"Job wrote {len(products)} product(s):\n")
for p in products:
    print(f"  - {p.type:10s}  name={p.name!r:40s}  file={p.file_id}  rev={p.revision_id}")

## 6 &middot; Search for a specific artifact (no model name or artifact ID required)

The sections above walked the job's own product list. In practice you often need to find an artifact **after the fact** — from a different script, a different session, or without any reference to the job that produced it.

Two approaches are shown below:

| Approach | When to use |
|---|---|
| `platform.resources()` filter | You know the exact filename, display name, or external identifier |
| `platform.client.search_resources()` | You have a keyword and want to search across all metadata fields |

Both return results platform-wide — **no model ID or artifact ID needed**.

### Approach A — structural filter via `platform.resources()`

`platform.resources()` returns a lazy `ResourceQuery`. Chain `.type("artifact")` to restrict to artifacts, then `.filter(file_name=...)` to match on filename. A single `file_name` value performs a SQL `LIKE` comparison, so partial names work.

Other useful filters: `display_name`, `description`, `version_name`, `external_identifier`, `mime_type`, `archive_status`.

In [ ]:
# Find all artifacts whose filename matches SEARCH_FILENAME.
# Nothing hits the network until .all() / .first() / iteration.
# file_name must be passed as a list (single value performs a LIKE comparison).
matches = (
    platform.resources()
    .type("artifact")
    .filter(file_name=[SEARCH_FILENAME])
    .all()
)

print(f"Found {len(matches)} artifact(s) named '{SEARCH_FILENAME}':\n")
for item in matches:
    print(f"  id={item.id}  name={item.name!r}  updated={item.updated}")

### Approach B — full-text search via `platform.client.search_resources()`

`search_resources` performs a full-text search across `name`, `description`, `display_name`, `version_name`, and `external_identifier` simultaneously. Use it when you have a keyword but don't know the exact filename.

Constraints: `search_term` ≥ 1 character; `page` ≥ 1; `size` between 1 and 100.

In [ ]:
from istari_digital_client.v2.models import FullTextSearch

search_results = platform.client.search_resources(
    full_text_search=FullTextSearch(
        search_term=SEARCH_KEYWORD,
        page=1,
        size=25,
    )
)

# Filter the results down to artifacts only
artifact_hits = [r for r in search_results.items if r.type_name == "artifact"]

print(f"Full-text search for '{SEARCH_KEYWORD}' returned {search_results.total} total hit(s).")
print(f"{len(artifact_hits)} of those are artifacts:\n")
for item in artifact_hits:
    print(f"  id={item.id}  name={item.name!r}  updated={item.updated}")

### Pick the target artifact and read its content

Both approaches above return `ResourceSearchItem` objects — lightweight index records with metadata but no file content. To read the actual bytes, fetch the full artifact and wrap it in a `ResourceView` (which provides `read_json()`, `read_text()`, and `download()`).

Here we prefer the most-recently-updated match from Approach A, falling back to Approach B if Approach A found nothing.

In [ ]:
# Pick the most-recently-updated artifact from whichever search found results.
candidates = matches or artifact_hits
assert candidates, (
    f"No artifacts found for filename='{SEARCH_FILENAME}' or keyword='{SEARCH_KEYWORD}'. "
    "Check that the extraction job completed and the search terms match."
)

# Sort by updated timestamp descending and take the newest.
target_item = sorted(candidates, key=lambda r: r.updated, reverse=True)[0]
print(f"Selected artifact: id={target_item.id}  name={target_item.name!r}")

# Fetch the full artifact object and wrap it in a ResourceView so we can read its content.
artifact_obj = platform.client.get_artifact(artifact_id=target_item.id)
artifact_view = ResourceView(_resource=artifact_obj, _client=platform.client)

print(f"\nResourceView: {artifact_view}")
print(f"Revision:     {artifact_view.revision_id}")

### Read the artifact content

`read_json()` downloads and parses the file in one call. Use `read_text()` for plain text or `download(dest)` to write to disk.

In [ ]:
requirements_data = artifact_view.read_json()

print(f"Parsed {len(requirements_data)} requirement(s):\n")
for req in requirements_data:
    print(f"  id={req['id']}")
    print(f"  name={req['name']}")
    print(f"  tags={req.get('tags', {})}")
    print()

## 7 &middot; Narrowing a search with multiple filters

When a keyword or filename matches many artifacts (e.g. across many model revisions), combine filters to zero in on the one you want.

All filter parameters that map to list-typed API fields (`file_name`, `display_name`, `external_identifier`, `version_name`, `mime_type`, etc.) **must be passed as Python lists**, even when filtering on a single value.

```python
# Narrow by filename AND external identifier
platform.resources()
    .type("artifact")
    .filter(
        file_name=["requirements.json"],
        external_identifier=["cameo-extract-and-search-demo"],
    )
    .first()

# Only artifacts you created, sorted newest first
platform.resources()
    .type("artifact")
    .filter(file_name=["requirements.json"])
    .sort("-created")
    .first()

# Full-text search restricted to a display name prefix
platform.resources()
    .type("artifact")
    .filter(display_name=["NCXTable"])
    .all()
```

Queries are lazy and immutable — each `.filter()` / `.sort()` / `.type()` call returns a new query object, so a base query can be safely forked and reused without side effects.

In [ ]:
# Example: find the requirements.json that belongs to the model we just uploaded,
# using external_identifier as the tie-breaker.
# All list-type filter params (file_name, display_name, external_identifier, ...) must be lists.
precise_match = (
    platform.resources()
    .type("artifact")
    .filter(
        file_name=[SEARCH_FILENAME],
        external_identifier=[EXTERNAL_ID],
    )
    .sort("-created")
    .first()
)

if precise_match:
    print(f"Precise match: id={precise_match.id}  name={precise_match.name!r}")
else:
    print(
        "No precise match found. The platform may not propagate external_identifier "
        "to job-produced artifacts automatically — use the filename-only filter instead."
    )

## Verify in the UI

Sign in to the same platform you used for your token and cross-check:

1. **Files / Models** — You should see the model with the display name set in Step 3.
2. **Jobs / Activity** — One `@istari:extract` job for `dassault_cameo`.
3. **Resources / Artifacts** — Search for `requirements.json` and confirm it appears with the correct revision.

## Optional &middot; Archive the model

Archiving hides the Model from default listings in the UI and API. The data and lineage stay intact — this is a soft delete you can reverse later.

In [ ]:
model.archive()